# Parameter Golf - TT+MLA+GQA

A100-optimized. Pre-tokenized pipeline, batch=1024, 10M+ tok/s.


## 1. Install


In [ ]:
!pip install -q torch transformers tokenizers datasets tqdm numpy

import torch
print(f"PyTorch {torch.__version__}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem/1e9:.1f} GB")


## 2. Clone


In [ ]:
!git clone -b tt-mla-gqa-submission https://github.com/ironbyte-rgb/parameter-golf.git /content/parameter-golf
%cd /content/parameter-golf
import sys; sys.path.insert(0, "/content/parameter-golf")
!ls *.py


## 3. Verify model


In [ ]:
from model import TTMLATransformer
import torch
m = TTMLATransformer().cuda()
print(f"Params: {sum(p.numel() for p in m.parameters()):,} ({sum(p.numel() for p in m.parameters())*2/1e6:.1f} MB BF16)")
x = torch.randint(0, 4096, (4, 512), device="cuda")
with torch.no_grad():
    loss = m(x, return_loss=True, targets=x.clone())
print(f"Init loss: {loss.item():.1f}")
del m, x; torch.cuda.empty_cache()
print("OK")


## 4. Train tokenizer (skip if tokenizer.json exists)


In [ ]:
from tokenizer_ import train_bpe_tokenizer, build_byte_luts
import os
if not os.path.exists("./tokenizer.json"):
    train_bpe_tokenizer(output_path="./tokenizer.json", vocab_size=4096)
from tokenizers import Tokenizer
tok = Tokenizer.from_file("./tokenizer.json")
print(f"Vocab: {tok.get_vocab_size()}")


## 5. Pre-tokenize data (~5-10 min, run once)


In [ ]:
!python prepare_data.py --num_tokens 2e9


## 6. Train


In [ ]:
%cd /content/parameter-golf
import os
os.environ["BATCH_SIZE"] = "1024"
os.environ["TRAIN_STEPS"] = "10000"
os.environ["LR"] = "6e-3"
os.environ["WARMUP_STEPS"] = "500"
os.environ["DECAY_STEPS"] = "1000"
os.environ["EVAL_EVERY"] = "1000"
os.environ["MAX_VAL_TOKENS"] = "200000"
os.environ["OUTPUT_DIR"] = "/content/parameter-golf/output"
print(f"Batch: {os.environ["BATCH_SIZE"]} | Tokens/step: {int(os.environ["BATCH_SIZE"]) * 512:,}")
from train_gpt import train, get_config
train(get_config())


## 7. Results


In [ ]:
%cd /content/parameter-golf
import os
for f in sorted(os.listdir("./output")):
    sz = os.path.getsize(f"./output/{f}") / 1e6
    print(f"  {f:40s} {sz:.2f} MB")
ptz = "./output/final_model.int8.ptz"
if os.path.exists(ptz):
    ptz_sz = os.path.getsize(ptz)
    code_sz = os.path.getsize("./train_gpt.py")
    total = ptz_sz + code_sz
    print(f"
Artifact: {total:,} bytes ({total/1e6:.2f} MB)  Under 16MB: {total < 16_000_000}")
